In [42]:
# ============================================================
# 040_weekly_papers_review
# ============================================================
#
# Overview
# ----------------
# Weekly aggregation and review of papers ingested in the last 7 days.
# Queries Notion Papers DB and RQ DB via Notion 2025+ "data_sources" API,
# ranks papers by importance and RQ relevance, applies READ/KEEP/SKIP decisions,
# and produces a curated weekly reading list + summary artifacts.
#
# This notebook is designed to remove noise and extract meaningful signals
# for a weekly "worldview update" workflow.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Notion Papers DB (NOTION_LIT_DB_ID): papers ingested in last 7 days
#   - Notion RQ DB (NOTION_RQ_DB_ID or RQ_DB_ID): research questions for relevance scoring
#   - config/rq_registry.yaml: fallback RQ registry if RQ DB unavailable
#   - decision_override.csv: optional human overrides for decisions
#   - env.txt: runtime config (NOTION_TOKEN, NOTION_VERSION, flags)
#
# Outputs:
#   - outputs/weekly/YYYY-Www/040_weekly_papers_review/weekly_papers_ranked.parquet|.csv
#   - outputs/weekly/YYYY-Www/040_weekly_papers_review/weekly_read_list.parquet|.csv
#   - outputs/weekly/YYYY-Www/040_weekly_papers_review/weekly_summary.md
#   - outputs/weekly/YYYY-Www/040_weekly_papers_review/weekly_papers_decisions.json
#   - outputs/weekly/YYYY-Www/040_weekly_papers_review/decision_override_template.csv
#
# Structure
# ----------------
# Cell 01: Imports, environment setup, and constants
# Cell 02: Notion client wrapper (request + retry + classification)
# Cell 03: Papers DB introspection (resolve data_source_id, infer property types)
# Cell 04: Fetch papers from last 7 days (data_sources query + pagination)
# Cell 05: Normalize papers to DataFrame + deduplication
# Cell 06: Fetch RQs and build in-memory registry (data_sources-first, YAML fallback)
# Cell 07: Compute importance and RQ relevance scores
# Cell 08: Apply decisions (adaptive percentile targets) + merge human overrides
# Cell 09: Generate outputs and weekly summary
# Cell 10: Notion writeback (optional; gated by env flags)
#
# Notes
# ----------------
# - Notion API (2025+):
#     * /databases/{db_id} is treated as a container (schema may be empty)
#     * Queries must use /data_sources/{data_source_id}/query
# - Schema handling:
#     * Property types are inferred from a sample page when DB schema is empty
#     * All writeback updates are schema-safe (only update properties that exist)
# - Dedup priority: Dedup Key > Source UID > normalized Name+PDF Link
# - Scoring: weekly_priority = 0.55*importance + 0.45*rq_relevance (configurable)
# - Decisions: percentile-based targets (stable week-to-week; avoids READ=0)
#     * READ: top READ_TOP_PCT (cap MAX_READ_COUNT), with adaptive min threshold
#     * KEEP: next band up to KEEP_TOP_PCT, with adaptive min threshold
# - Writeback:
#     * ENABLE_NOTION_WRITEBACK=true to PATCH results back to Notion
#     * ENABLE_NOTION_WRITEBACK_DRYRUN=true to test without writing
# - Each cell kept short (~20–60 lines); complex logic moved into helper functions


In [31]:
# ============================================================
# Cell 01 — Imports, environment setup, and constants
# ============================================================
# Overview:
#   Load environment variables, configure runtime LLM settings,
#   import standard libraries, and define constants used across cells.
#
# Inputs / Outputs:
#   Inputs: env.txt (environment variables)
#   Outputs: Configured environment, imported modules, global constants
#
# Notes:
#   - NOTION_TOKEN, NOTION_LIT_DB_ID, NOTION_RQ_DB_ID loaded from env.txt
#   - Uses iso8601week for YYYY-Www date formatting
#   - Output paths follow outputs/weekly/YYYY-Www/040_weekly_papers_review/

import os
import json
import math
import time
import logging
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, List, Any, Optional

import pandas as pd
import numpy as np
import yaml
import requests
from dotenv import load_dotenv

# ------------------------------------------------------------
# Environment variables
# ------------------------------------------------------------
load_dotenv("env.txt")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_VERSION = os.getenv("NOTION_VERSION", "2025-09-03")

NOTION_LIT_DB_ID = os.getenv("NOTION_LIT_DB_ID")
PAPERS_DB_ID = NOTION_LIT_DB_ID
NOTION_RQ_DB_ID = os.getenv("NOTION_RQ_DB_ID", "")
MONITORING_QUEUE_DB_ID = os.getenv("NOTION_MONITORING_QUEUE_DB_ID", "")

ENABLE_NOTION_WRITEBACK = os.getenv("ENABLE_NOTION_WRITEBACK", "false").lower() == "true"

if not NOTION_TOKEN or not PAPERS_DB_ID:
    raise RuntimeError("Missing NOTION_TOKEN or NOTION_LIT_DB_ID in env.txt")

# ------------------------------------------------------------
# Time & weekly context
# ------------------------------------------------------------
NOW_UTC = datetime.now(timezone.utc)
WEEK_START = NOW_UTC - timedelta(days=7)
START_DATE = WEEK_START.strftime("%Y-%m-%d")
END_DATE = NOW_UTC.strftime("%Y-%m-%d")

ISO_YEAR, ISO_WEEK, _ = NOW_UTC.isocalendar()
WEEK_ID = f"{ISO_YEAR}-W{ISO_WEEK:02d}"
week_str = WEEK_ID

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------
BASE_OUTPUT_DIR = Path("outputs") / "weekly" / WEEK_ID / "040_weekly_papers_review"
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DECISION_OVERRIDE_PATH = BASE_OUTPUT_DIR / "decision_override.csv"
RQ_REGISTRY_PATH = Path("config") / "rq_registry.yaml"

# ------------------------------------------------------------
# Scoring & decision constants (tunable)
# ------------------------------------------------------------
IMPORTANCE_WEIGHT = 0.55
RQ_RELEVANCE_WEIGHT = 0.45

READ_THRESHOLD = 65
KEEP_THRESHOLD = 40
MAX_READ_COUNT = 20

READ_TOP_PCT = 0.20   # 上位20%をREAD候補
KEEP_TOP_PCT = 0.60   # 上位60%までをKEEP以上（つまり次の40%がKEEP）
READ_MIN_SCORE = 45   # 読むには最低これくらいは欲しい（保険）
KEEP_MIN_SCORE = 25   # KEEPの最低（保険）
MAX_READ_COUNT = 20   # 既存の上限制約

# ------------------------------------------------------------
# Notion API constants
# ------------------------------------------------------------
NOTION_BASE_URL = "https://api.notion.com/v1"

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("weekly_papers_review")

logger.info(f"Weekly Papers Review initialized for {WEEK_ID}")
logger.info(f"Output directory: {BASE_OUTPUT_DIR}")
logger.info(f"Date range: {START_DATE} to {END_DATE}")
logger.info(f"Notion writeback: {'ENABLED' if ENABLE_NOTION_WRITEBACK else 'DISABLED'}")


2026-02-06 06:54:49,578 | INFO | Weekly Papers Review initialized for 2026-W06
2026-02-06 06:54:49,579 | INFO | Output directory: outputs/weekly/2026-W06/040_weekly_papers_review
2026-02-06 06:54:49,580 | INFO | Date range: 2026-01-29 to 2026-02-05
2026-02-06 06:54:49,580 | INFO | Notion writeback: ENABLED


In [33]:
# ============================================================
# Cell 02 — Notion client wrapper with REST API and pagination
# ============================================================
# Overview:
#   Robust Notion REST API client:
#   - Single entrypoint request() with rate limit + retry + error classification
#   - Helper methods for database schema retrieval and paginated queries
#
# Inputs / Outputs:
#   Inputs: NOTION_TOKEN, NOTION_VERSION
#   Outputs: notion_client with request(), get_database_schema(), query_database()
#
# Notes:
#   - Uses requests.Session for connection pooling
#   - Handles pagination via has_more and next_cursor
#   - Use request(debug=True) to inspect raw responses when diagnosing

import time
import requests
from enum import Enum
from typing import Optional, Dict, Any, Tuple, List
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

class ErrorCategory(Enum):
    RETRYABLE = "retryable"      # 429, 5xx, transient network
    MANUAL_FIX = "manual_fix"    # 400 validation/schema/property mismatch
    FATAL = "fatal"              # 401/403 auth, 404 not found
    SUCCESS = "success"

class NotionAPIError(RuntimeError):
    def __init__(self, message: str, category: ErrorCategory, status_code: Optional[int] = None, payload: Optional[dict] = None):
        super().__init__(message)
        self.category = category
        self.status_code = status_code
        self.payload = payload or {}

class NotionClient:
    def __init__(
        self,
        token: str,
        notion_version: str,
        base_url: str = "https://api.notion.com/v1",
        timeout: int = 30,
        max_retries: int = 3,
        min_request_interval: float = 0.34,
    ):
        if not token:
            raise ValueError("NOTION_TOKEN is required")

        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.max_retries = max_retries
        self.min_request_interval = min_request_interval
        self._last_request_time = 0.0

        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Notion-Version": notion_version,
                "Content-Type": "application/json",
            }
        )

        adapter = HTTPAdapter(max_retries=Retry(total=0, raise_on_status=False))
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def _enforce_rate_limit(self):
        now = time.time()
        dt = now - self._last_request_time
        if dt < self.min_request_interval:
            time.sleep(self.min_request_interval - dt)
        self._last_request_time = time.time()

    def _classify_error(self, status_code: int, error_json: Optional[dict]) -> Tuple[ErrorCategory, str]:
        msg = (error_json or {}).get("message", "") if isinstance(error_json, dict) else ""
        code = (error_json or {}).get("code", "unknown") if isinstance(error_json, dict) else "unknown"

        if status_code in (401, 403):
            return ErrorCategory.FATAL, f"Auth/permission error ({status_code}) code={code} msg={msg}"
        if status_code == 404:
            return ErrorCategory.FATAL, f"Not found ({status_code}) code={code} msg={msg}"
        if status_code == 400:
            return ErrorCategory.MANUAL_FIX, f"Validation/schema error ({status_code}) code={code} msg={msg}"
        if status_code == 429 or status_code >= 500:
            return ErrorCategory.RETRYABLE, f"Retryable error ({status_code}) code={code} msg={msg}"
        if 400 <= status_code < 500:
            return ErrorCategory.MANUAL_FIX, f"Client error ({status_code}) code={code} msg={msg}"
        return ErrorCategory.FATAL, f"Unexpected error ({status_code}) code={code} msg={msg}"

    def request(self, method: str, path: str, json: Optional[Dict[str, Any]] = None, debug: bool = False) -> Dict[str, Any]:
        method = method.upper()
        url = f"{self.base_url}{path}"
        backoff = 1.0

        for attempt in range(1, self.max_retries + 2):
            try:
                self._enforce_rate_limit()
                resp = self.session.request(method=method, url=url, json=json, timeout=self.timeout)

                if debug:
                    print("\n[HTTP DEBUG]")
                    print("method:", method, "url:", url, "status:", resp.status_code)
                    print("content-type:", resp.headers.get("content-type"))
                    print("raw text (first 400):", resp.text[:400])

                if resp.status_code in (200, 201):
                    return resp.json()

                err_json = None
                try:
                    err_json = resp.json()
                except Exception:
                    err_json = {"message": resp.text}

                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = float(retry_after) if retry_after else backoff
                    logger.warning("429 rate limited. wait=%.1fs attempt=%d path=%s", wait, attempt, path)
                    time.sleep(wait)
                    backoff *= 2
                    continue

                if 500 <= resp.status_code < 600:
                    logger.warning("5xx server error %d. backoff=%.1fs attempt=%d path=%s", resp.status_code, backoff, attempt, path)
                    time.sleep(backoff)
                    backoff *= 2
                    continue

                category, msg = self._classify_error(resp.status_code, err_json)
                raise NotionAPIError(msg, category=category, status_code=resp.status_code, payload=err_json)

            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt <= self.max_retries + 1:
                    logger.warning("Network error. backoff=%.1fs attempt=%d path=%s", backoff, attempt, path)
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                raise NotionAPIError(f"Network error after retries: {e}", category=ErrorCategory.RETRYABLE)

        raise NotionAPIError("Unexpected failure in NotionClient.request()", category=ErrorCategory.FATAL)

    def get_database_schema(self, database_id: str, debug: bool = False) -> Dict[str, Any]:
        return self.request("GET", f"/databases/{database_id}", debug=debug)

    def query_database(self, database_id: str, payload: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        results: List[Dict[str, Any]] = []
        body = payload.copy() if payload else {}

        while True:
            data = self.request("POST", f"/databases/{database_id}/query", json=body)
            results.extend(data.get("results", []))
            if not data.get("has_more"):
                break
            body["start_cursor"] = data.get("next_cursor")

        logger.info("Fetched %d pages from database %s", len(results), database_id)
        return results

PAPERS_SCHEMA = {
    "database_id": PAPERS_DB_ID if "PAPERS_DB_ID" in globals() else NOTION_LIT_DB_ID,
    "required_properties": {
        # --- Existing (from LIT_SCHEMA) ---
        "Name": "title",
        "Created time": "created_time",
        "Authors & Year": "rich_text",
        "Tags": "multi_select",
        "PDF Link": "url",
        "Findings": "rich_text",
        "Core Idea": "rich_text",
        "Notes": "rich_text",
        "Methods": "rich_text",
        "Type": "select",
        "Source": "select",
        "Datasets": "rich_text",
        "Papers": "relation",

        # --- Plan A additions ---
        "Status": "select",
        "Dedup Key": "rich_text",
        "Source UID": "rich_text",
        "Ingested At": "date",
        "Run ID": "rich_text",
        "PDF Status": "select",
        "Slide 1 URL": "url",

        # --- Weekly scoring/writeback additions (NEW) ---
        "Importance": "number",
        "RQ Relevance": "number",
        "Weekly Priority": "number",
        "Decision": "select",
        "Decision Reason": "rich_text",
    },
    "optional_properties": {},
}

notion_client = NotionClient(token=NOTION_TOKEN, notion_version=NOTION_VERSION)
logger.info("NotionClient ready (rate limit + retry + classification)")


2026-02-06 06:58:27,249 | INFO | NotionClient ready (rate limit + retry + classification)


In [34]:
# ============================================================
# Cell 03 — Schema validation and property introspection
# ============================================================
# Overview:
#   Notion 2025+ style:
#   - Query is performed via /data_sources/{data_source_id}/query
#   This cell:
#   1) retrieves database metadata
#   2) picks a data_source id
#   3) queries 1 row to infer property names/types
#   4) validates required properties (from PAPERS_SCHEMA if available, else fallback)
#
# Inputs / Outputs:
#   Inputs: NOTION_LIT_DB_ID, notion_client
#   Outputs:
#     - papers_db_meta (dict)
#     - papers_data_source_id (str)
#     - papers_property_types (dict): {prop_name: notion_type}
#     - papers_property_map (dict): {canonical: actual} (exact-match)
#
# Notes:
#   - Uses NOTION_LIT_DB_ID as requested.
#   - If PAPERS_SCHEMA is not defined yet, uses a small fallback required set.

import re
from typing import Dict, Any

def extract_32hex(value: str) -> str:
    if not value:
        return ""
    s = value.replace("-", "")
    m = re.search(r"([0-9a-fA-F]{32})", s)
    return m.group(1).lower() if m else ""

def to_hyphen_uuid(hex32: str) -> str:
    if not hex32 or len(hex32) != 32:
        return ""
    return f"{hex32[0:8]}-{hex32[8:12]}-{hex32[12:16]}-{hex32[16:20]}-{hex32[20:32]}"

# --- Normalize DB ID (use NOTION_LIT_DB_ID as requested) ---
raw_db_id = NOTION_LIT_DB_ID
hex32 = extract_32hex(raw_db_id)
db_id = to_hyphen_uuid(hex32) if hex32 else ""
if not db_id:
    raise RuntimeError("NOTION_LIT_DB_ID must contain a 32-hex database id (or a Notion URL including it).")

logger.info(f"Papers DB ID (raw): {raw_db_id}")
logger.info(f"Papers DB ID (hex32): {hex32}")
logger.info(f"Papers DB ID (uuid): {db_id}")

# --- Retrieve database metadata ---
papers_db_meta = notion_client.request("GET", f"/databases/{db_id}", debug=True)
if papers_db_meta.get("object") != "database":
    raise RuntimeError(f"ID does not resolve to a database. id={db_id} object={papers_db_meta.get('object')}")

# --- Pick data_source id ---
data_sources = papers_db_meta.get("data_sources") or []
if not data_sources:
    raise RuntimeError(
        "No data_sources found on database object. "
        "Set NOTION_VERSION=2025-09-03 (or latest) in env.txt and retry."
    )

papers_data_source_id = data_sources[0]["id"]
logger.info(f"Using papers_data_source_id: {papers_data_source_id} (name={data_sources[0].get('name')})")

# --- Query 1 row to infer property names/types ---
q = notion_client.request(
    "POST",
    f"/data_sources/{papers_data_source_id}/query",
    json={"page_size": 1},
    debug=True,
)

rows = q.get("results", []) or []
if not rows:
    raise RuntimeError("Papers data source returned 0 rows; cannot infer schema from sample page.")

sample_props = rows[0].get("properties") or {}
papers_property_types: Dict[str, str] = {}
for prop_name, prop_obj in sample_props.items():
    ptype = (prop_obj or {}).get("type")
    if ptype:
        papers_property_types[prop_name] = ptype

logger.info(f"Inferred {len(papers_property_types)} properties from sample page.")
logger.info(f"Inferred properties: {list(papers_property_types.keys())}")

# --- Build required property list ---
if 'PAPERS_SCHEMA' in globals() and PAPERS_SCHEMA:
    required_names = list(PAPERS_SCHEMA.get('required_properties', {}).keys())
else:
    required_names = [
        'Name', 'Created time', 'Authors & Year', 'Tags', 'PDF Link',
        'Findings', 'Core Idea', 'Notes', 'Methods', 'Type', 'Source',
        'Datasets', 'Papers', 'Status', 'Dedup Key', 'Source UID',
        'Ingested At', 'Run ID', 'PDF Status', 'Slide 1 URL'
    ]

logger.info(f"Required properties to validate: {required_names}")

# --- Validate and build property map (exact match only) ---
papers_property_map = {}
missing_props = []

for canonical in required_names:
    if canonical in papers_property_types:
        papers_property_map[canonical] = canonical
    else:
        missing_props.append(canonical)

if missing_props:
    logger.warning(f"Missing properties in schema: {missing_props}")
else:
    logger.info("All required properties found in schema.")

papers_property_map = {p: p for p in required_names if p in papers_property_types}
logger.info(f"Property map built with {len(papers_property_map)} mappings.")
logger.info(f"Property map: {papers_property_map}")


2026-02-06 06:58:31,218 | INFO | Papers DB ID (raw): 2a98e0e4d16280cbb6cbdcd1ebedee54
2026-02-06 06:58:31,220 | INFO | Papers DB ID (hex32): 2a98e0e4d16280cbb6cbdcd1ebedee54
2026-02-06 06:58:31,221 | INFO | Papers DB ID (uuid): 2a98e0e4-d162-80cb-b6cb-dcd1ebedee54
2026-02-06 06:58:31,538 | INFO | Using papers_data_source_id: 2a98e0e4-d162-801c-87a6-000bc077f4ff (name=Literature Database)



[HTTP DEBUG]
method: GET url: https://api.notion.com/v1/databases/2a98e0e4-d162-80cb-b6cb-dcd1ebedee54 status: 200
content-type: application/json; charset=utf-8
raw text (first 400): {"object":"database","id":"2a98e0e4-d162-80cb-b6cb-dcd1ebedee54","title":[{"type":"text","text":{"content":"Literature Database","link":null},"annotations":{"bold":false,"italic":false,"strikethrough":false,"underline":false,"code":false,"color":"default"},"plain_text":"Literature Database","href":null}],"description":[],"parent":{"type":"page_id","page_id":"2a88e0e4-d162-80f4-8ed6-e0083e9259e0"},


2026-02-06 06:58:33,046 | INFO | Inferred 25 properties from sample page.
2026-02-06 06:58:33,047 | INFO | Inferred properties: ['Created time', 'Decision', 'PDF Status', 'Authors & Year', 'Slide 1 URL', 'Dedup Key', 'Source UID', 'Type', 'Methods', 'Notes', 'Importance', 'Tags', 'PDF Link', 'Datasets', 'Source', 'Findings', 'Weekly Priority', 'Run ID', 'Papers', 'RQ Relevance', 'Decision Reason', 'Status', 'Ingested At', 'Core Idea', 'Name']
2026-02-06 06:58:33,048 | INFO | Required properties to validate: ['Name', 'Created time', 'Authors & Year', 'Tags', 'PDF Link', 'Findings', 'Core Idea', 'Notes', 'Methods', 'Type', 'Source', 'Datasets', 'Papers', 'Status', 'Dedup Key', 'Source UID', 'Ingested At', 'Run ID', 'PDF Status', 'Slide 1 URL', 'Importance', 'RQ Relevance', 'Weekly Priority', 'Decision', 'Decision Reason']
2026-02-06 06:58:33,049 | INFO | All required properties found in schema.
2026-02-06 06:58:33,051 | INFO | Property map built with 25 mappings.
2026-02-06 06:58:33,052 


[HTTP DEBUG]
method: POST url: https://api.notion.com/v1/data_sources/2a98e0e4-d162-801c-87a6-000bc077f4ff/query status: 200
content-type: application/json; charset=utf-8
raw text (first 400): {"object":"list","results":[{"object":"page","id":"2132b6ad-fd39-4f9f-a433-45a2fbdbd855","created_time":"2025-11-15T22:35:00.000Z","last_edited_time":"2025-11-15T22:42:00.000Z","created_by":{"object":"user","id":"a5ef9484-ac87-4c5d-92bb-c308b7d743d6"},"last_edited_by":{"object":"user","id":"a5ef9484-ac87-4c5d-92bb-c308b7d743d6"},"cover":null,"icon":null,"parent":{"type":"data_source_id","data_sour


In [35]:
# ============================================================
# Cell 04 — Fetch papers from last 7 days
# ============================================================
# Overview:
#   Query Notion data source for papers ingested in the last 7 days.
#   Uses /data_sources/{data_source_id}/query endpoint (Notion 2025+).
#
# Inputs / Outputs:
#   Inputs: papers_data_source_id, papers_property_map, notion_client, WEEK_START
#   Outputs: raw_papers (list of page objects)
#
# Notes:
#   - Uses papers_data_source_id from Cell 03 schema introspection
#   - Filters on 'Ingested At' property if available
#   - Falls back to 'Created time' if 'Ingested At' not found
#   - Date filter: last 7 days from WEEK_START

from datetime import datetime, timedelta, timezone

# --- Verify required variables from Cell 03 ---
if not papers_data_source_id:
    raise RuntimeError("papers_data_source_id not set. Run Cell 03 first.")

if not papers_property_map:
    raise RuntimeError("papers_property_map is empty. Run Cell 03 first.")

logger.info(f"Using data_source_id: {papers_data_source_id}")
logger.info(f"Property map has {len(papers_property_map)} properties.")

# --- Determine date filter property ---
date_prop_name = None

if 'Ingested At' in papers_property_map:
    date_prop_name = papers_property_map['Ingested At']
    logger.info(f"Using 'Ingested At' property for date filter: {date_prop_name}")
elif 'Created time' in papers_property_map:
    date_prop_name = papers_property_map['Created time']
    logger.info(f"Fallback to 'Created time' property for date filter: {date_prop_name}")
else:
    logger.warning("No date property found. Fetching all papers (no date filter).")

# --- Build query payload ---
query_payload = {"page_size": 100}

if date_prop_name:
    seven_days_ago = WEEK_START.isoformat()
    query_payload["filter"] = {
        "property": date_prop_name,
        "date": {
            "on_or_after": seven_days_ago
        }
    }
    query_payload["sorts"] = [
        {
            "property": date_prop_name,
            "direction": "descending"
        }
    ]
    logger.info(f"Date filter: on_or_after {seven_days_ago}")

# --- Query data source ---
logger.info("Querying Notion data source for papers...")

raw_papers = []
body = query_payload.copy()

while True:
    resp = notion_client.request(
        "POST",
        f"/data_sources/{papers_data_source_id}/query",
        json=body
    )
    
    results = resp.get("results", []) or []
    raw_papers.extend(results)
    
    if not resp.get("has_more"):
        break
    
    body["start_cursor"] = resp.get("next_cursor")
    logger.info(f"Fetched {len(results)} pages, continuing pagination...")

logger.info(f"Fetched {len(raw_papers)} papers from data source.")

if len(raw_papers) == 0:
    logger.warning("No papers found matching date filter. Check 'Ingested At' property values.")
else:
    logger.info(f"Sample paper ID: {raw_papers[0].get('id')}")

# --- raw_papers is now ready for normalization in Cell 05 ---


2026-02-06 06:58:35,190 | INFO | Using data_source_id: 2a98e0e4-d162-801c-87a6-000bc077f4ff
2026-02-06 06:58:35,191 | INFO | Property map has 25 properties.
2026-02-06 06:58:35,192 | INFO | Using 'Ingested At' property for date filter: Ingested At
2026-02-06 06:58:35,194 | INFO | Date filter: on_or_after 2026-01-29T21:54:49.575952+00:00
2026-02-06 06:58:35,195 | INFO | Querying Notion data source for papers...
2026-02-06 06:58:35,621 | INFO | Fetched 47 papers from data source.
2026-02-06 06:58:35,622 | INFO | Sample paper ID: 2fe8e0e4-d162-81e9-bdac-da8f313da079


In [36]:
# ============================================================
# Cell 05 — Normalize papers to DataFrame with deduplication
# ============================================================
# Overview:
#   Parse raw Notion page objects into a structured DataFrame.
#   Extract all mapped properties with safe fallbacks for missing values.
#   Apply deduplication logic: Dedup Key > Source UID > normalized Name+PDF Link.
#   Retain earliest ingested paper when duplicates detected.
#
# Inputs / Outputs:
#   Inputs: raw_papers (list of Notion page dicts), papers_property_map
#   Outputs: papers_df (pandas DataFrame with deduplicated papers)
#
# Notes:
#   - Handles missing properties gracefully (returns None/empty string)
#   - Date Ingested parsed to datetime for sorting
#   - Deduplication priority: Dedup Key > Source UID > Name+PDF Link hash
#   - Keeps earliest ingested paper within each duplicate group
#   - Logs deduplication summary and sample records

def extract_property_value(page: Dict, prop_name: str) -> Any:
    """Extract property value from Notion page object with type handling."""
    properties = page.get('properties', {})
    if prop_name not in properties:
        return None
    
    prop_data = properties[prop_name]
    prop_type = prop_data.get('type')
    
    if prop_type == 'title':
        titles = prop_data.get('title', [])
        return titles[0].get('plain_text', '') if titles else ''
    
    elif prop_type == 'rich_text':
        texts = prop_data.get('rich_text', [])
        return texts[0].get('plain_text', '') if texts else ''
    
    elif prop_type == 'url':
        return prop_data.get('url', '')
    
    elif prop_type == 'date':
        date_obj = prop_data.get('date')
        if date_obj:
            return date_obj.get('start', '')
        return None
    
    elif prop_type == 'number':
        return prop_data.get('number')
    
    elif prop_type == 'select':
        select_obj = prop_data.get('select')
        return select_obj.get('name', '') if select_obj else ''
    
    elif prop_type == 'multi_select':
        multi = prop_data.get('multi_select', [])
        return ', '.join([item.get('name', '') for item in multi])
    
    elif prop_type == 'created_time':
        return prop_data.get('created_time', '')
    
    elif prop_type == 'relation':
        relations = prop_data.get('relation', [])
        return ', '.join([rel.get('id', '') for rel in relations])
    
    else:
        return None

# --- Parse raw papers into structured records ---
logger.info("Normalizing papers to DataFrame...")

paper_records = []
for page in raw_papers:
    record = {
        'notion_page_id': page.get('id', ''),
        'notion_url': page.get('url', ''),
    }
    
    # Extract all mapped properties
    for canonical_name, notion_prop_name in papers_property_map.items():
        value = extract_property_value(page, notion_prop_name)
        record[canonical_name] = value
    
    paper_records.append(record)

# --- Create DataFrame ---
papers_df = pd.DataFrame(paper_records)

logger.info(f"Parsed {len(papers_df)} paper records.")

# --- Convert Ingested At to datetime ---
if 'Ingested At' in papers_df.columns:
    papers_df['Ingested At'] = pd.to_datetime(papers_df['Ingested At'], errors='coerce')
    papers_df = papers_df.sort_values('Ingested At', ascending=True)
else:
    logger.warning("Ingested At column not found; skipping date parsing.")

# --- Deduplication logic ---
logger.info("Applying deduplication logic...")

# Create normalized dedup key fallback: lowercase name + pdf link
if 'Name' in papers_df.columns and 'PDF Link' in papers_df.columns:
    papers_df['_dedup_fallback'] = (
        papers_df['Name'].fillna('').str.lower().str.strip() + '||' +
        papers_df['PDF Link'].fillna('').str.lower().str.strip()
    )
else:
    papers_df['_dedup_fallback'] = ''

# Build final dedup key with priority
def build_dedup_key(row):
    if 'Dedup Key' in row and pd.notna(row.get('Dedup Key')) and row.get('Dedup Key') != '':
        return f"dedup:{row['Dedup Key']}"
    elif 'Source UID' in row and pd.notna(row.get('Source UID')) and row.get('Source UID') != '':
        return f"uid:{row['Source UID']}"
    else:
        return f"fallback:{row['_dedup_fallback']}"

papers_df['_final_dedup_key'] = papers_df.apply(build_dedup_key, axis=1)

# Remove records with empty dedup key (no usable identifier)
papers_df = papers_df[papers_df['_final_dedup_key'] != 'fallback:||'].copy()

original_count = len(papers_df)

# Keep earliest ingested paper per dedup key
papers_df = papers_df.drop_duplicates(subset='_final_dedup_key', keep='first')

deduped_count = len(papers_df)
duplicates_removed = original_count - deduped_count

logger.info(f"Deduplication complete: {original_count} -> {deduped_count} papers ({duplicates_removed} duplicates removed).")

# --- Clean up temporary columns ---
papers_df = papers_df.drop(columns=['_dedup_fallback', '_final_dedup_key'])

# --- Log sample ---
if len(papers_df) > 0:
    logger.info(f"Sample paper: {papers_df.iloc[0]['Name'][:50]}... (ID: {papers_df.iloc[0]['notion_page_id']})")
    logger.info(f"Columns: {list(papers_df.columns)}")
else:
    logger.warning("No papers remaining after deduplication.")

# --- papers_df is now ready for scoring ---


2026-02-06 06:58:37,726 | INFO | Normalizing papers to DataFrame...
2026-02-06 06:58:37,737 | INFO | Parsed 47 paper records.
2026-02-06 06:58:37,755 | INFO | Applying deduplication logic...
2026-02-06 06:58:37,773 | INFO | Deduplication complete: 47 -> 47 papers (0 duplicates removed).
2026-02-06 06:58:37,775 | INFO | Sample paper: Beyond Blink: A Challenge to Behavioral Decision M... (ID: 2f88e0e4-d162-81a8-9f3e-ddbdfc9134bd)
2026-02-06 06:58:37,776 | INFO | Columns: ['notion_page_id', 'notion_url', 'Name', 'Created time', 'Authors & Year', 'Tags', 'PDF Link', 'Findings', 'Core Idea', 'Notes', 'Methods', 'Type', 'Source', 'Datasets', 'Papers', 'Status', 'Dedup Key', 'Source UID', 'Ingested At', 'Run ID', 'PDF Status', 'Slide 1 URL', 'Importance', 'RQ Relevance', 'Weekly Priority', 'Decision', 'Decision Reason']


In [37]:
# ============================================================
# Cell 06 — Fetch RQs and build in-memory registry (data_sources-first)
# ============================================================
# Overview:
#   Build RQ registry primarily from Notion RQ DB using Notion 2025+ data_sources API.
#   Fallback to local YAML/JSON registry if RQ DB is unavailable or empty.
#
# Inputs / Outputs:
#   Inputs: RQ_DB_ID (or NOTION_RQ_DB_ID), RQ_SCHEMA (optional), notion_client
#   Outputs: rq_registry (list of dict: {id, text, keywords, weight, meta})
#
# Notes:
#   - Notion 2025+: query must use /data_sources/{data_source_id}/query
#   - Minimal: id + text + keywords. weight defaults to 1.0
#   - If Status property exists, filters to Active-ish values when possible

import re
from pathlib import Path

RQ_REGISTRY_PATH = Path("config/rq_registry.yaml")  # optional fallback

def extract_32hex(value: str) -> str:
    if not value:
        return ""
    s = value.replace("-", "")
    m = re.search(r"([0-9a-fA-F]{32})", s)
    return m.group(1).lower() if m else ""

def to_hyphen_uuid(hex32: str) -> str:
    if not hex32 or len(hex32) != 32:
        return ""
    return f"{hex32[0:8]}-{hex32[8:12]}-{hex32[12:16]}-{hex32[16:20]}-{hex32[20:32]}"

def get_data_source_id_from_db(db_id_uuid: str) -> str:
    db_meta = notion_client.request("GET", f"/databases/{db_id_uuid}")
    ds = db_meta.get("data_sources") or []
    return ds[0]["id"] if ds else ""

def prop_text(page: dict, prop_name: str) -> str:
    """Extract human text from common Notion property types (best-effort)."""
    p = (page.get("properties") or {}).get(prop_name) or {}
    t = p.get("type")

    if t == "title":
        return "".join(x.get("plain_text", "") for x in (p.get("title") or []))
    if t == "rich_text":
        return "".join(x.get("plain_text", "") for x in (p.get("rich_text") or []))
    if t == "select":
        sel = p.get("select") or {}
        return sel.get("name", "") or ""
    if t == "multi_select":
        ms = p.get("multi_select") or []
        return ", ".join(x.get("name", "") for x in ms if x.get("name"))
    if t == "number":
        return "" if p.get("number") is None else str(p.get("number"))
    if t == "checkbox":
        return "true" if p.get("checkbox") else "false"
    return ""

def normalize_keywords(*parts: str) -> str:
    s = " ".join([p for p in parts if p]).strip()
    return re.sub(r"\s+", " ", s)

rq_registry = []

# Prefer RQ_DB_ID if present, else NOTION_RQ_DB_ID if defined in globals
rq_db_raw = RQ_DB_ID if "RQ_DB_ID" in globals() and RQ_DB_ID else (globals().get("NOTION_RQ_DB_ID") or "")
if rq_db_raw:
    rq_db_hex = extract_32hex(rq_db_raw)
    rq_db_id = to_hyphen_uuid(rq_db_hex) if rq_db_hex else ""
else:
    rq_db_id = ""

if rq_db_id:
    logger.info(f"Fetching RQs from Notion RQ DB (uuid): {rq_db_id}")

    try:
        rq_ds_id = get_data_source_id_from_db(rq_db_id)
        if not rq_ds_id:
            raise RuntimeError("No data_sources found on RQ database. Check NOTION_VERSION>=2025-09-03.")

        # Query payload: fetch up to 100; filter optional (Status) is applied only if schema is known
        payload = {"page_size": 100}

        # If RQ_SCHEMA exists and includes status property name, try an Active filter (best-effort)
        status_prop_name = ""
        title_prop_name = "Name"

        if "RQ_SCHEMA" in globals() and isinstance(RQ_SCHEMA, dict):
            title_prop_name = RQ_SCHEMA.get("title", "Name") or "Name"
            status_prop_name = RQ_SCHEMA.get("status", "") or ""

        # NOTE: Notion filter syntax depends on property type. We'll attempt select-based filter only.
        if status_prop_name:
            payload["filter"] = {
                "property": status_prop_name,
                "select": {"equals": "Active"},
            }

        data = notion_client.request("POST", f"/data_sources/{rq_ds_id}/query", json=payload)
        pages = data.get("results", []) or []
        logger.info(f"Retrieved {len(pages)} RQs from Notion (pre-filter).")

        # If filter didn't work (or status isn't select), we will post-filter by text
        for pg in pages:
            rq_text = prop_text(pg, title_prop_name).strip()
            if not rq_text:
                continue

            # Optional fields (best-effort; derive keywords from tags/title/gap/rationale)
            tags_name = ""
            gap_name = ""
            rationale_name = ""
            priority_name = ""

            if "RQ_SCHEMA" in globals() and isinstance(RQ_SCHEMA, dict):
                tags_name = RQ_SCHEMA.get("tags", "") or ""
                gap_name = RQ_SCHEMA.get("gap", "") or ""
                rationale_name = RQ_SCHEMA.get("rationale", "") or ""
                priority_name = RQ_SCHEMA.get("priority", "") or ""

            status_val = prop_text(pg, status_prop_name).strip() if status_prop_name else ""
            # post-filter "active-ish" if status exists but filter failed
            if status_val and status_val.lower() not in ("active", "in progress", "doing", "open"):
                continue

            tags_val = prop_text(pg, tags_name) if tags_name else ""
            gap_val = prop_text(pg, gap_name) if gap_name else ""
            rationale_val = prop_text(pg, rationale_name) if rationale_name else ""

            # weight: use numeric priority if available, else default 1.0
            weight = 1.0
            if priority_name:
                pr = prop_text(pg, priority_name)
                try:
                    weight = float(pr) if pr else 1.0
                except Exception:
                    weight = 1.0

            rq_registry.append(
                {
                    "id": pg.get("id", ""),
                    "text": rq_text,
                    "keywords": normalize_keywords(rq_text, tags_val, gap_val, rationale_val),
                    "weight": weight,
                    "meta": {"status": status_val, "tags": tags_val},
                }
            )

        logger.info(f"Built RQ registry with {len(rq_registry)} active-ish RQs from Notion.")

    except Exception as e:
        logger.warning(f"Failed to fetch RQs from Notion: {e}. Falling back to local registry.")
        rq_registry = []

# --- Fallback to YAML if Notion fetch failed or empty ---
if len(rq_registry) == 0:
    logger.info(f"Falling back to local RQ registry: {RQ_REGISTRY_PATH}")

    try:
        import yaml  # optional dependency
    except Exception:
        yaml = None

    if RQ_REGISTRY_PATH.exists() and yaml is not None:
        with open(RQ_REGISTRY_PATH, "r", encoding="utf-8") as f:
            items = yaml.safe_load(f)

        if isinstance(items, list):
            for rq in items:
                text = (rq.get("text") or "").strip()
                if not text:
                    continue
                rq_registry.append(
                    {
                        "id": rq.get("id", ""),
                        "text": text,
                        "keywords": (rq.get("keywords") or text).strip(),
                        "weight": float(rq.get("weight", 1.0)),
                        "meta": {},
                    }
                )
            logger.info(f"Loaded {len(rq_registry)} RQs from YAML.")
        else:
            logger.warning("YAML RQ registry is empty or malformed.")
    else:
        if yaml is None:
            logger.warning("PyYAML not installed; cannot load YAML fallback.")
        else:
            logger.warning(f"YAML RQ registry not found at {RQ_REGISTRY_PATH}.")

if len(rq_registry) == 0:
    logger.error("No research questions available. RQ relevance scoring will be skipped.")
else:
    logger.info(f"Final RQ registry contains {len(rq_registry)} research questions.")
    logger.info(f"Sample RQ: {rq_registry[0]['text'][:80]}...")


2026-02-06 06:58:39,600 | INFO | Fetching RQs from Notion RQ DB (uuid): 2a98e0e4-d162-80b7-bad2-cf6635a3ef17
2026-02-06 06:58:40,530 | INFO | Retrieved 60 RQs from Notion (pre-filter).
2026-02-06 06:58:40,532 | INFO | Built RQ registry with 60 active-ish RQs from Notion.
2026-02-06 06:58:40,533 | INFO | Final RQ registry contains 60 research questions.
2026-02-06 06:58:40,534 | INFO | Sample RQ: どの公的介入が民間ベンチャーキャピタルの呼び込みに最も効果的か？...


In [38]:
# ============================================================
# Cell 07 — Compute importance and RQ relevance scores
# ============================================================
# Overview:
#   Compute importance score (0-100) based on paper metadata (authors, abstract, source).
#   Compute RQ relevance score (0-100) by matching paper content against RQ registry.
#   Combine scores into weekly_priority = 0.55*importance + 0.45*rq_relevance.
#   Use LLM for semantic scoring; fallback to heuristics if LLM unavailable.
#
# Inputs / Outputs:
#   Inputs: papers_df, rq_registry, llm_provider, llm_model, llm_temperature
#   Outputs: papers_df with added columns: importance_score, rq_relevance_score, weekly_priority
#
# Notes:
#   - Importance heuristics: has authors (+20), has abstract (+30), known source (+20), long abstract (+30)
#   - RQ relevance heuristics: keyword overlap with RQ keywords, text similarity to RQ text
#   - LLM scoring uses structured prompts for consistency
#   - Missing LLM falls back to heuristics with warning
#   - Scores normalized to 0-100 range

# --- Initialize scoring columns ---
papers_df['importance_score'] = 0.0
papers_df['rq_relevance_score'] = 0.0
papers_df['weekly_priority'] = 0.0

logger.info("Computing importance and RQ relevance scores...")

# --- Helper: Heuristic importance scoring ---
def compute_importance_heuristic(row: pd.Series) -> float:
    """
    Compute importance score based on metadata completeness and quality signals.
    Returns score in range 0-100.
    """
    score = 0.0
    
    # Has authors
    if 'Authors & Year' in row and pd.notna(row.get('Authors & Year')) and row.get('Authors & Year', '').strip() != '':
        score += 20.0
    
    # Has core idea or findings (abstract equivalents)
    abstract_fields = ['Core Idea', 'Findings', 'Notes']
    has_content = False
    content_length = 0
    
    for field in abstract_fields:
        if field in row and pd.notna(row.get(field)) and row.get(field, '').strip() != '':
            has_content = True
            content_length += len(str(row.get(field, '')).strip())
    
    if has_content:
        score += 30.0
        # Bonus for substantial content (>500 chars suggests comprehensive description)
        if content_length > 500:
            score += 10.0
    
    # Known/trusted source
    source = str(row.get('Source', '')).lower()
    trusted_sources = ['arxiv', 'pubmed', 'semantic scholar', 'google scholar', 'acm', 'ieee']
    if any(ts in source for ts in trusted_sources):
        score += 20.0
    
    # Has PDF link
    if 'PDF Link' in row and pd.notna(row.get('PDF Link')) and row.get('PDF Link', '').strip() != '':
        score += 20.0
    
    return min(score, 100.0)

# --- Helper: Heuristic RQ relevance scoring ---
def compute_rq_relevance_heuristic(row: pd.Series, rq_registry: List[Dict]) -> float:
    """
    Compute RQ relevance score based on keyword/text overlap with RQs.
    Returns score in range 0-100.
    """
    if len(rq_registry) == 0:
        return 0.0
    
    # Build paper text for matching (name + content fields + tags)
    paper_text_parts = []
    
    if 'Name' in row and pd.notna(row.get('Name')):
        paper_text_parts.append(str(row['Name']).lower())
    
    # Collect content from various fields
    content_fields = ['Core Idea', 'Findings', 'Notes', 'Methods']
    for field in content_fields:
        if field in row and pd.notna(row.get(field)):
            paper_text_parts.append(str(row[field]).lower())
    
    # Extract keywords from tags
    if 'Tags' in row and pd.notna(row.get('Tags')):
        paper_text_parts.append(str(row['Tags']).lower())
    
    paper_text = ' '.join(paper_text_parts)
    
    if not paper_text.strip():
        return 0.0
    
    # Compute weighted overlap with each RQ
    total_weight = sum(rq.get('weight', 1.0) for rq in rq_registry)
    weighted_score = 0.0
    
    for rq in rq_registry:
        rq_text = rq.get('text', '').lower()
        rq_keywords = rq.get('keywords', '').lower()
        rq_weight = rq.get('weight', 1.0)
        
        match_score = 0.0
        
        # Check for RQ text words in paper text
        rq_words = set(rq_text.split())
        rq_words = {w for w in rq_words if len(w) > 3}  # Filter short words
        if rq_words:
            matches = sum(1 for word in rq_words if word in paper_text)
            match_score += (matches / len(rq_words)) * 50.0
        
        # Check for RQ keyword matches
        if rq_keywords.strip():
            kw_list = [kw.strip() for kw in rq_keywords.split(',')]
            kw_matches = sum(1 for kw in kw_list if kw in paper_text)
            if kw_list:
                match_score += (kw_matches / len(kw_list)) * 50.0
        
        # Weight and accumulate
        weighted_score += (match_score * rq_weight)
    
    # Normalize by total weight
    final_score = (weighted_score / total_weight) if total_weight > 0 else 0.0
    return min(final_score, 100.0)

# --- Compute scores for all papers ---
for idx, row in papers_df.iterrows():
    # Compute importance
    importance = compute_importance_heuristic(row)
    papers_df.at[idx, 'importance_score'] = importance
    
    # Compute RQ relevance
    rq_relevance = compute_rq_relevance_heuristic(row, rq_registry)
    papers_df.at[idx, 'rq_relevance_score'] = rq_relevance
    
    # Compute weekly priority
    weekly_priority = (
        IMPORTANCE_WEIGHT * importance +
        RQ_RELEVANCE_WEIGHT * rq_relevance
    )
    papers_df.at[idx, 'weekly_priority'] = weekly_priority

logger.info("Scoring complete.")

# --- Log scoring statistics ---
if len(papers_df) > 0:
    logger.info(f"Importance scores: min={papers_df['importance_score'].min():.1f}, "
                f"max={papers_df['importance_score'].max():.1f}, "
                f"mean={papers_df['importance_score'].mean():.1f}")
    
    logger.info(f"RQ relevance scores: min={papers_df['rq_relevance_score'].min():.1f}, "
                f"max={papers_df['rq_relevance_score'].max():.1f}, "
                f"mean={papers_df['rq_relevance_score'].mean():.1f}")
    
    logger.info(f"Weekly priority scores: min={papers_df['weekly_priority'].min():.1f}, "
                f"max={papers_df['weekly_priority'].max():.1f}, "
                f"mean={papers_df['weekly_priority'].mean():.1f}")
    
    # Sort by weekly_priority descending
    papers_df = papers_df.sort_values('weekly_priority', ascending=False).reset_index(drop=True)
    
    logger.info(f"Top paper by priority: {papers_df.iloc[0]['Name'][:60]}... "
                f"(priority={papers_df.iloc[0]['weekly_priority']:.1f})")
else:
    logger.warning("No papers to score.")

# --- papers_df is now scored and sorted by priority ---


2026-02-06 06:58:40,633 | INFO | Computing importance and RQ relevance scores...
2026-02-06 06:58:40,662 | INFO | Scoring complete.
2026-02-06 06:58:40,663 | INFO | Importance scores: min=40.0, max=80.0, mean=55.1
2026-02-06 06:58:40,664 | INFO | RQ relevance scores: min=0.0, max=0.2, mean=0.0
2026-02-06 06:58:40,665 | INFO | Weekly priority scores: min=22.0, max=44.1, mean=30.3
2026-02-06 06:58:40,667 | INFO | Top paper by priority: Solving the Problem of Abundance: Venture Capital and the Ma... (priority=44.1)


In [39]:
# ============================================================
# Cell 08 — Apply decisions and merge human overrides
# ============================================================
# Overview:
#   Apply READ/KEEP/SKIP decisions using percentile-based targets.
#   Make thresholds adaptive to weekly score distribution so READ doesn't go to zero.
#   Then merge human overrides and generate template CSV.
#
# Inputs / Outputs:
#   Inputs: papers_df, DECISION_OVERRIDE_PATH, BASE_OUTPUT_DIR
#   Outputs: papers_df['decision'], decision_override_template.csv
#
# Notes:
#   - READ: top READ_TOP_PCT (cap MAX_READ_COUNT) AND score >= adaptive_read_min
#   - KEEP: top KEEP_TOP_PCT AND score >= adaptive_keep_min
#   - adaptive mins are derived from cutoffs with a margin, with floors.

READ_TOP_PCT = globals().get("READ_TOP_PCT", 0.20)
KEEP_TOP_PCT = globals().get("KEEP_TOP_PCT", 0.60)
MAX_READ_COUNT = globals().get("MAX_READ_COUNT", 20)

# Floors (absolute minimum guards) — loosen these if you want more
READ_FLOOR = globals().get("READ_FLOOR", 20)
KEEP_FLOOR = globals().get("KEEP_FLOOR", 10)

# How much below the cutoff we still allow
READ_MARGIN = globals().get("READ_MARGIN", 10)   # was implicitly 0 before (too strict)
KEEP_MARGIN = globals().get("KEEP_MARGIN", 8)

logger.info("Applying automated decision logic (adaptive percentile-based)...")

papers_df["weekly_priority"] = pd.to_numeric(papers_df["weekly_priority"], errors="coerce").fillna(0.0)
papers_df = papers_df.sort_values("weekly_priority", ascending=False).reset_index(drop=True)

papers_df["decision"] = "SKIP"
papers_df["decision_reason"] = "Below KEEP band or score too low"

n = len(papers_df)
if n == 0:
    logger.warning("papers_df is empty; skipping decision logic.")
else:
    read_n = min(MAX_READ_COUNT, max(1, int(math.ceil(n * READ_TOP_PCT))))
    keep_n = max(read_n, int(math.ceil(n * KEEP_TOP_PCT)))
    keep_n = min(keep_n, n)

    read_cut = float(papers_df.loc[read_n - 1, "weekly_priority"]) if read_n >= 1 else float("inf")
    keep_cut = float(papers_df.loc[keep_n - 1, "weekly_priority"]) if keep_n >= 1 else float("inf")

    # Adaptive minimums: follow the week's distribution
    adaptive_read_min = max(READ_FLOOR, read_cut - READ_MARGIN)
    adaptive_keep_min = max(KEEP_FLOOR, keep_cut - KEEP_MARGIN)

    logger.info("Decision targets: READ=%d (cap %d), KEEP<=%d (n=%d)", read_n, MAX_READ_COUNT, keep_n, n)
    logger.info("Decision cutoffs: read_cut=%.1f keep_cut=%.1f", read_cut, keep_cut)
    logger.info("Adaptive mins: read_min=%.1f (floor=%d, margin=%d), keep_min=%.1f (floor=%d, margin=%d)",
                adaptive_read_min, READ_FLOOR, READ_MARGIN, adaptive_keep_min, KEEP_FLOOR, KEEP_MARGIN)

    for i, row in papers_df.iterrows():
        s = float(row["weekly_priority"])

        if i < read_n and s >= adaptive_read_min:
            papers_df.at[i, "decision"] = "READ"
            papers_df.at[i, "decision_reason"] = f"Top {READ_TOP_PCT:.0%} band + score>=adaptive_read_min({adaptive_read_min:.1f})"
        elif i < keep_n and s >= adaptive_keep_min:
            papers_df.at[i, "decision"] = "KEEP"
            papers_df.at[i, "decision_reason"] = f"Top {KEEP_TOP_PCT:.0%} band + score>=adaptive_keep_min({adaptive_keep_min:.1f})"
        else:
            papers_df.at[i, "decision"] = "SKIP"
            papers_df.at[i, "decision_reason"] = "Below KEEP band or score too low"

decision_counts = papers_df["decision"].value_counts().to_dict()
logger.info("Automated decisions: READ=%d, KEEP=%d, SKIP=%d",
            decision_counts.get("READ", 0),
            decision_counts.get("KEEP", 0),
            decision_counts.get("SKIP", 0))

# --- Human overrides (same as before) ---
if DECISION_OVERRIDE_PATH.exists():
    logger.info(f"Loading human decision overrides from {DECISION_OVERRIDE_PATH}...")
    try:
        overrides_df = pd.read_csv(DECISION_OVERRIDE_PATH)
        required_cols = ["notion_page_id", "decision"]
        if not all(col in overrides_df.columns for col in required_cols):
            logger.warning(f"Override CSV missing required columns: {required_cols}. Skipping overrides.")
        else:
            override_count = 0
            overrides_df["decision"] = overrides_df["decision"].astype(str).str.strip().str.upper()
            if "reason" not in overrides_df.columns:
                overrides_df["reason"] = ""

            for _, r in overrides_df.iterrows():
                page_id = str(r["notion_page_id"]).strip()
                new_decision = r["decision"]
                reason = (str(r.get("reason", "")) or "Human override").strip()

                if not page_id or new_decision not in ["READ", "KEEP", "SKIP"]:
                    continue

                mask = papers_df["notion_page_id"].astype(str) == page_id
                if mask.any():
                    old = papers_df.loc[mask, "decision"].iloc[0]
                    papers_df.loc[mask, "decision"] = new_decision
                    papers_df.loc[mask, "decision_reason"] = f"Human override: {reason}"
                    override_count += 1
                    logger.info("Override applied: %s %s -> %s", page_id, old, new_decision)

            logger.info(f"Applied {override_count} human overrides.")
            decision_counts = papers_df["decision"].value_counts().to_dict()
            logger.info("Final decisions: READ=%d, KEEP=%d, SKIP=%d",
                        decision_counts.get("READ", 0),
                        decision_counts.get("KEEP", 0),
                        decision_counts.get("SKIP", 0))
    except Exception as e:
        logger.error(f"Failed to load overrides: {e}")
else:
    logger.info("No decision override file found. Using automated decisions only.")

# --- Template (same as before) ---
template_path = BASE_OUTPUT_DIR / "decision_override_template.csv"
logger.info(f"Generating decision override template: {template_path}")

template_df = papers_df[["notion_page_id", "Name", "decision", "weekly_priority", "decision_reason"]].copy()
template_df = template_df.rename(columns={
    "Name": "paper_name",
    "decision": "current_decision",
    "weekly_priority": "priority_score",
})
template_df.insert(3, "decision", "")
template_df.insert(4, "reason", "")
template_df.to_csv(template_path, index=False)
logger.info(f"Template saved with {len(template_df)} papers.")

logger.info("Decision application complete.")


2026-02-06 06:58:46,066 | INFO | Applying automated decision logic (adaptive percentile-based)...
2026-02-06 06:58:46,070 | INFO | Decision targets: READ=10 (cap 20), KEEP<=29 (n=47)
2026-02-06 06:58:46,071 | INFO | Decision cutoffs: read_cut=38.5 keep_cut=22.0
2026-02-06 06:58:46,072 | INFO | Adaptive mins: read_min=28.5 (floor=20, margin=10), keep_min=14.0 (floor=10, margin=8)
2026-02-06 06:58:46,078 | INFO | Automated decisions: READ=10, KEEP=19, SKIP=18
2026-02-06 06:58:46,080 | INFO | No decision override file found. Using automated decisions only.
2026-02-06 06:58:46,080 | INFO | Generating decision override template: outputs/weekly/2026-W06/040_weekly_papers_review/decision_override_template.csv
2026-02-06 06:58:46,085 | INFO | Template saved with 47 papers.
2026-02-06 06:58:46,086 | INFO | Decision application complete.


In [40]:
# ============================================================
# Cell 09 — Generate outputs and weekly summary
# ============================================================
# Overview:
#   Export all papers with scores and decisions to parquet and CSV.
#   Generate curated weekly reading list (READ papers only).
#   Create structured weekly summary markdown report.
#   Export decision metadata as JSON.
#
# Inputs / Outputs:
#   Inputs: papers_df (with scores and decisions), BASE_OUTPUT_DIR, week_str
#   Outputs:
#     - weekly_papers_ranked.parquet|.csv: all papers with scores
#     - weekly_read_list.parquet|.csv: READ papers only
#     - weekly_summary.md: markdown report
#     - weekly_papers_decisions.json: decision metadata
#
# Notes:
#   - All outputs saved to outputs/weekly/YYYY-Www/040_weekly_papers_review/
#   - Summary includes statistics, top papers, decision breakdown
#   - Reading list sorted by priority descending
#   - JSON includes decision counts and timestamp

logger.info("Generating output files...")

# --- 1. Export full ranked papers list ---
ranked_papers_parquet = BASE_OUTPUT_DIR / 'weekly_papers_ranked.parquet'
ranked_papers_csv = BASE_OUTPUT_DIR / 'weekly_papers_ranked.csv'

papers_df.to_parquet(ranked_papers_parquet, index=False)
papers_df.to_csv(ranked_papers_csv, index=False)

logger.info(f"Saved ranked papers: {ranked_papers_parquet} ({len(papers_df)} papers)")

# --- 2. Generate weekly reading list (READ papers only) ---
read_list_df = papers_df[papers_df['decision'] == 'READ'].copy()
read_list_df = read_list_df.sort_values('weekly_priority', ascending=False).reset_index(drop=True)

read_list_parquet = BASE_OUTPUT_DIR / 'weekly_read_list.parquet'
read_list_csv = BASE_OUTPUT_DIR / 'weekly_read_list.csv'

if len(read_list_df) > 0:
    read_list_df.to_parquet(read_list_parquet, index=False)
    read_list_df.to_csv(read_list_csv, index=False)
    logger.info(f"Saved reading list: {read_list_parquet} ({len(read_list_df)} papers)")
else:
    logger.warning("No READ papers to export. Reading list will be empty.")
    pd.DataFrame().to_parquet(read_list_parquet, index=False)
    pd.DataFrame().to_csv(read_list_csv, index=False)

# --- 3. Generate decision metadata JSON ---
decision_counts = papers_df['decision'].value_counts().to_dict()

decision_metadata = {
    'week': week_str,
    'generated_at': datetime.now().isoformat(),
    'total_papers': len(papers_df),
    'decision_counts': {
        'READ': decision_counts.get('READ', 0),
        'KEEP': decision_counts.get('KEEP', 0),
        'SKIP': decision_counts.get('SKIP', 0)
    },
    'scoring_config': {
        'importance_weight': IMPORTANCE_WEIGHT,
        'rq_relevance_weight': RQ_RELEVANCE_WEIGHT,
        'read_threshold': READ_THRESHOLD,
        'keep_threshold': KEEP_THRESHOLD,
        'max_read_count': MAX_READ_COUNT
    },
    'score_statistics': {
        'weekly_priority': {
            'min': float(papers_df['weekly_priority'].min()) if len(papers_df) > 0 else 0.0,
            'max': float(papers_df['weekly_priority'].max()) if len(papers_df) > 0 else 0.0,
            'mean': float(papers_df['weekly_priority'].mean()) if len(papers_df) > 0 else 0.0
        },
        'importance': {
            'min': float(papers_df['importance_score'].min()) if len(papers_df) > 0 else 0.0,
            'max': float(papers_df['importance_score'].max()) if len(papers_df) > 0 else 0.0,
            'mean': float(papers_df['importance_score'].mean()) if len(papers_df) > 0 else 0.0
        },
        'rq_relevance': {
            'min': float(papers_df['rq_relevance_score'].min()) if len(papers_df) > 0 else 0.0,
            'max': float(papers_df['rq_relevance_score'].max()) if len(papers_df) > 0 else 0.0,
            'mean': float(papers_df['rq_relevance_score'].mean()) if len(papers_df) > 0 else 0.0
        }
    }
}

decision_json_path = BASE_OUTPUT_DIR / 'weekly_papers_decisions.json'
with open(decision_json_path, 'w', encoding='utf-8') as f:
    json.dump(decision_metadata, f, indent=2, ensure_ascii=False)

logger.info(f"Saved decision metadata: {decision_json_path}")

# --- 4. Generate weekly summary markdown ---
summary_md_path = BASE_OUTPUT_DIR / 'weekly_summary.md'

logger.info(f"Generating weekly summary: {summary_md_path}")

with open(summary_md_path, 'w', encoding='utf-8') as f:
    f.write(f"# Weekly Papers Review — {week_str}\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("## Overview\n\n")
    f.write(f"- **Papers ingested:** {len(papers_df)}\n")
    f.write(f"- **Date range:** {START_DATE} to {END_DATE}\n")
    f.write(f"- **Research questions active:** {len(rq_registry)}\n\n")
    
    f.write("## Decision Summary\n\n")
    f.write(f"| Decision | Count | Percentage |\n")
    f.write(f"|----------|-------|------------|\n")
    total = len(papers_df) if len(papers_df) > 0 else 1
    for decision in ['READ', 'KEEP', 'SKIP']:
        count = decision_counts.get(decision, 0)
        pct = (count / total) * 100
        f.write(f"| {decision} | {count} | {pct:.1f}% |\n")
    f.write("\n")
    
    f.write("## Score Statistics\n\n")
    if len(papers_df) > 0:
        f.write(f"- **Weekly Priority:** min={papers_df['weekly_priority'].min():.1f}, "
                f"max={papers_df['weekly_priority'].max():.1f}, "
                f"mean={papers_df['weekly_priority'].mean():.1f}\n")
        f.write(f"- **Importance:** min={papers_df['importance_score'].min():.1f}, "
                f"max={papers_df['importance_score'].max():.1f}, "
                f"mean={papers_df['importance_score'].mean():.1f}\n")
        f.write(f"- **RQ Relevance:** min={papers_df['rq_relevance_score'].min():.1f}, "
                f"max={papers_df['rq_relevance_score'].max():.1f}, "
                f"mean={papers_df['rq_relevance_score'].mean():.1f}\n\n")
    else:
        f.write("No papers to analyze.\n\n")
    
    f.write("## Reading List (READ Decision)\n\n")
    if len(read_list_df) > 0:
        f.write(f"**{len(read_list_df)} papers recommended for reading this week:**\n\n")
        for idx, row in read_list_df.iterrows():
            f.write(f"### {idx + 1}. {row['Name']}\n\n")
            f.write(f"- **Priority Score:** {row['weekly_priority']:.1f} "
                    f"(Importance: {row['importance_score']:.1f}, RQ Relevance: {row['rq_relevance_score']:.1f})\n")
            
            if 'Authors & Year' in row and pd.notna(row['Authors & Year']) and row['Authors & Year'] != '':
                f.write(f"- **Authors:** {row['Authors & Year']}\n")
            
            if 'Source' in row and pd.notna(row['Source']) and row['Source'] != '':
                f.write(f"- **Source:** {row['Source']}\n")
            
            if 'PDF Link' in row and pd.notna(row['PDF Link']) and row['PDF Link'] != '':
                f.write(f"- **PDF:** [{row['PDF Link']}]({row['PDF Link']})\n")
            
            if 'notion_url' in row and pd.notna(row['notion_url']):
                f.write(f"- **Notion:** [{row['notion_url']}]({row['notion_url']})\n")
            
            abstract_text = ''
            for field in ['Core Idea', 'Findings', 'Notes']:
                if field in row and pd.notna(row[field]) and row[field] != '':
                    abstract_text = str(row[field])
                    break
            
            if abstract_text:
                abstract_preview = abstract_text[:300]
                if len(abstract_text) > 300:
                    abstract_preview += '...'
                f.write(f"- **Abstract:** {abstract_preview}\n")
            
            f.write("\n")
    else:
        f.write("No papers recommended for reading this week.\n\n")
    
    f.write("## Top Papers by Priority (All Decisions)\n\n")
    top_n = min(10, len(papers_df))
    if top_n > 0:
        f.write(f"**Top {top_n} papers by weekly priority:**\n\n")
        f.write("| Rank | Name | Priority | Decision |\n")
        f.write("|------|------|----------|----------|\n")
        for i, (idx, row) in enumerate(papers_df.head(top_n).iterrows(), 1):
            name_short = row['Name'][:60]
            if len(row['Name']) > 60:
                name_short += '...'
            f.write(f"| {i} | {name_short} | {row['weekly_priority']:.1f} | {row['decision']} |\n")
        f.write("\n")
    
    f.write("## Research Questions Used\n\n")
    if len(rq_registry) > 0:
        for i, rq in enumerate(rq_registry, 1):
            f.write(f"{i}. {rq['text']} (weight: {rq['weight']})\n")
    else:
        f.write("No research questions configured.\n")
    f.write("\n")
    
    f.write("## Output Files\n\n")
    f.write(f"- `weekly_papers_ranked.parquet|.csv`: All {len(papers_df)} papers with scores and decisions\n")
    f.write(f"- `weekly_read_list.parquet|.csv`: {len(read_list_df)} papers marked for reading\n")
    f.write(f"- `weekly_papers_decisions.json`: Decision metadata and statistics\n")
    f.write(f"- `decision_override_template.csv`: Template for manual decision overrides\n")
    f.write(f"- `weekly_summary.md`: This summary report\n\n")
    
    f.write("---\n\n")
    f.write("*Generated by 040_weekly_papers_review notebook*\n")

logger.info(f"Summary saved: {summary_md_path}")

# --- Log completion ---
logger.info("=" * 60)
logger.info("Weekly papers review complete!")
logger.info(f"Output directory: {BASE_OUTPUT_DIR}")
logger.info(f"Papers reviewed: {len(papers_df)}")
logger.info(f"Papers to read: {len(read_list_df)}")
logger.info(f"Decision breakdown: {decision_counts}")
logger.info("=" * 60)


2026-02-06 06:58:48,707 | INFO | Generating output files...
2026-02-06 06:58:48,729 | INFO | Saved ranked papers: outputs/weekly/2026-W06/040_weekly_papers_review/weekly_papers_ranked.parquet (47 papers)
2026-02-06 06:58:48,745 | INFO | Saved reading list: outputs/weekly/2026-W06/040_weekly_papers_review/weekly_read_list.parquet (10 papers)
2026-02-06 06:58:48,747 | INFO | Saved decision metadata: outputs/weekly/2026-W06/040_weekly_papers_review/weekly_papers_decisions.json
2026-02-06 06:58:48,748 | INFO | Generating weekly summary: outputs/weekly/2026-W06/040_weekly_papers_review/weekly_summary.md
2026-02-06 06:58:48,755 | INFO | Summary saved: outputs/weekly/2026-W06/040_weekly_papers_review/weekly_summary.md
2026-02-06 06:58:48,756 | INFO | ============================================================
2026-02-06 06:58:48,757 | INFO | Weekly papers review complete!
2026-02-06 06:58:48,757 | INFO | Output directory: outputs/weekly/2026-W06/040_weekly_papers_review
2026-02-06 06:58:48,7

In [41]:
# ============================================================
# Cell 10 — Notion writeback (safe, data_sources-compatible)
# ============================================================
# Overview:
#   Write computed scores/decisions back to Notion Papers pages (PATCH /pages/{id}).
#   Safe-by-default:
#     - ENABLE_NOTION_WRITEBACK=true to actually write
#     - ENABLE_NOTION_WRITEBACK_DRYRUN=true to log payloads without PATCH
#
# Inputs / Outputs:
#   Inputs:
#     - papers_df (must include notion_page_id, decision, decision_reason, weekly_priority, importance_score, rq_relevance_score)
#     - papers_property_types (from Cell 03; inferred from sample page)
#     - ENABLE_NOTION_WRITEBACK, ENABLE_NOTION_WRITEBACK_DRYRUN
#   Outputs:
#     - Notion pages updated (if enabled)
#
# Notes:
#   - Updates only properties that exist in papers_property_types (i.e., real DB schema)
#   - Handles missing properties gracefully (logs and skips)
#   - Rate-limited (~3 req/sec)

import time
from typing import Dict, Any

ENABLE_NOTION_WRITEBACK = str(os.getenv("ENABLE_NOTION_WRITEBACK", "false")).lower() == "true"
ENABLE_NOTION_WRITEBACK_DRYRUN = str(os.getenv("ENABLE_NOTION_WRITEBACK_DRYRUN", "false")).lower() == "true"

def build_prop_value(prop_type: str, value: Any) -> Dict[str, Any]:
    """Build Notion property payload for PATCH /pages/{id}."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        # For most property types, setting null clears; but Notion differs by type.
        # We'll just skip null updates upstream to avoid accidental clears.
        return {}

    if prop_type == "number":
        return {"number": float(value)}
    if prop_type == "select":
        return {"select": {"name": str(value)}}
    if prop_type == "rich_text":
        s = str(value)
        return {"rich_text": [{"text": {"content": s[:2000]}}]}
    if prop_type == "title":
        s = str(value)
        return {"title": [{"text": {"content": s[:2000]}}]}
    if prop_type == "url":
        return {"url": str(value)}
    # You can extend as needed (date, multi_select, etc.)
    return {}

def patch_page(page_id: str, props: Dict[str, Any]) -> bool:
    try:
        if ENABLE_NOTION_WRITEBACK_DRYRUN:
            logger.info("[DRYRUN] PATCH %s props=%s", page_id, list(props.keys()))
            return True
        notion_client.request("PATCH", f"/pages/{page_id}", json={"properties": props})
        return True
    except Exception as e:
        logger.error("Failed to update page %s: %s", page_id, str(e)[:200])
        return False

def writeback_papers(papers_df: pd.DataFrame) -> Dict[str, int]:
    if not ENABLE_NOTION_WRITEBACK:
        logger.info("Notion writeback disabled (ENABLE_NOTION_WRITEBACK=false). Skipping.")
        return {"success": 0, "failed": 0, "skipped": len(papers_df)}

    if "papers_property_types" not in globals() or not isinstance(papers_property_types, dict) or len(papers_property_types) == 0:
        raise RuntimeError("papers_property_types not available. Ensure Cell 03 ran successfully before writeback.")

    # Map DataFrame columns -> Notion property names (exact match by default)
    # Only properties that actually exist in papers_property_types will be updated.
    CANDIDATE_UPDATES = [
        # (df_col, notion_prop_name, fallback_type_if_unknown)
        ("importance_score", "Importance", "number"),
        ("rq_relevance_score", "RQ Relevance", "number"),
        ("weekly_priority", "Weekly Priority", "number"),
        ("decision", "Decision", "select"),
        ("decision_reason", "Decision Reason", "rich_text"),
    ]

    # Resolve what is writable (exists in real schema)
    writable = []
    for df_col, notion_name, fallback_type in CANDIDATE_UPDATES:
        if df_col not in papers_df.columns:
            continue
        if notion_name not in papers_property_types:
            logger.warning("Writeback property not found in DB schema: '%s' (skipping)", notion_name)
            continue
        writable.append((df_col, notion_name, papers_property_types.get(notion_name, fallback_type)))

    logger.info("Writable properties: %s", [(n, t) for _, n, t in writable])

    success = failed = skipped = 0

    for i, row in papers_df.iterrows():
        page_id = row.get("notion_page_id")
        if not page_id or pd.isna(page_id):
            skipped += 1
            continue

        props_payload: Dict[str, Any] = {}
        for df_col, notion_name, notion_type in writable:
            v = row.get(df_col)
            pv = build_prop_value(notion_type, v)
            if pv:
                props_payload[notion_name] = pv

        if not props_payload:
            skipped += 1
            continue

        ok = patch_page(page_id, props_payload)
        if ok:
            success += 1
        else:
            failed += 1

        time.sleep(0.35)  # ~3 req/sec

    logger.info("Writeback complete: success=%d failed=%d skipped=%d", success, failed, skipped)
    return {"success": success, "failed": failed, "skipped": skipped}

# --- Execute if enabled ---
if ENABLE_NOTION_WRITEBACK:
    logger.info("Notion writeback ENABLED%s.", " (DRYRUN)" if ENABLE_NOTION_WRITEBACK_DRYRUN else "")
    writeback_results = writeback_papers(papers_df)
    logger.info("=" * 60)
    logger.info("Notion Writeback Summary: %s", writeback_results)
    logger.info("=" * 60)
else:
    logger.info("Notion writeback DISABLED. Set ENABLE_NOTION_WRITEBACK=true in env.txt to enable.")
    logger.info("Tip: set ENABLE_NOTION_WRITEBACK_DRYRUN=true to test without updating Notion.")


2026-02-06 06:58:50,232 | INFO | Notion writeback ENABLED.
2026-02-06 06:58:50,233 | INFO | Writable properties: [('Importance', 'number'), ('RQ Relevance', 'number'), ('Weekly Priority', 'number'), ('Decision', 'select'), ('Decision Reason', 'rich_text')]
2026-02-06 06:59:32,177 | INFO | Writeback complete: success=47 failed=0 skipped=0
2026-02-06 06:59:32,178 | INFO | ============================================================
2026-02-06 06:59:32,179 | INFO | Notion Writeback Summary: {'success': 47, 'failed': 0, 'skipped': 0}
2026-02-06 06:59:32,181 | INFO | ============================================================
